# Heat Equation Inverse with DeepXDE
I attempt to solve an inverse problem of the Heat Equation with DeepXDE.

In [1]:
import deepxde as dde
import numpy as np
import torch

No backend selected.
Finding available backend...
Found pytorch
Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting the default backend to "pytorch". You can change it in the ~/.deepxde/config.json file or export the DDE_BACKEND environment variable. Valid options are: tensorflow.compat.v1, tensorflow, pytorch, jax, paddle (all lowercase)


In [2]:
# define the domain
geom = dde.geometry.Interval(0.0, 1.0) # so x is from 0 to 1
timedomain = dde.geometry.TimeDomain(0.0, 1.0) # so t is from 0 to 1
geomtime = dde.geometry.GeometryXTime(geom, timedomain) # space time domain

In [ ]:
# define the PDE

# unknown parameter alpha
alpha = dde.Variable(0.5)

# define the PDE residual
def heat_pde(x, u):
    # note that x has shape (N, 2) where x[:,0] is space and x[:,1] is time
    u_t = dde.grad.jacobian(u, x, i=0, j=1)  # du/dt (j = 1 implies time derivative)
    u_xx = dde.grad.hessian(u, x, i=0, j=0)  # d2u/dx2 (j = 0 implies space derivative)
    # note also that first order uses jacobian, second order uses hessian
    return u_t - alpha * u_xx

# define the initial condition
def initial_condition(x):
    return np.sin(np.pi * x[:, 0:1])  # u(x,0) = sin(pi*x)

ic = dde.icbc.IC(
    geomtime,
    initial_condition,
    lambda _, on_initial: on_initial
)

# boundary conditions
# these are like the edges of the rod, we set them to zero to indicate they are kept at 0 degrees
bc_left = dde.icbc.DirichletBC(
    geomtime,
    lambda x: 0.0,
    lambda x, on_boundary: on_boundary and np.isclose(x[0], 0.0)
)

bc_right = dde.icbc.DirichletBC(
    geomtime,
    lambda x: 0.0,
    lambda x, on_boundary: on_boundary and np.isclose(x[0], 1.0)
)



In [31]:
# Optional observational data

# True (unknown-to-model) diffusivity used only for data generation
alpha_true = 0.1

def true_solution(x, t):
    return np.sin(np.pi * x) * np.exp(-alpha_true * np.pi**2 * t)

# Number of observation points (sensors)
N = 100

# Sample random space-time locations
x_obs = np.random.rand(N, 1)   # x in [0, 1]
t_obs = np.random.rand(N, 1)   # t in [0, 1]

# Combine into DeepXDE input format: (x, t)
X_obs = np.hstack((x_obs, t_obs))  # shape (N, 2)

# Generate observed temperatures
u_obs = true_solution(X_obs[:, 0:1], X_obs[:, 1:2])  # shape (N, 1)

# Optional measurement noise
noise_std = 0.01
u_obs += noise_std * np.random.randn(*u_obs.shape)

# DeepXDE observation constraint
observe_u = dde.icbc.PointSetBC(X_obs, u_obs)


In [33]:
# data object
data = dde.data.TimePDE(
    geomtime,
    heat_pde,
    [ic, bc_left, bc_right, observe_u],
    num_domain=10000,
    num_boundary=200,
    num_initial=200
)

# neural network
net = dde.nn.FNN(
    [2] + [50] * 3 + [1],
    activation="tanh",
    kernel_initializer="Glorot normal"
)

In [35]:
model = dde.Model(data, net)

model.compile(
    optimizer="adam",
    lr=1e-3,
    external_trainable_variables=[alpha]
)

Compiling model...
'compile' took 0.000326 s



In [36]:
model.train(epochs=5000)

print("Learned alpha:", alpha.item())

Training model...

0         [4.79e-02, 5.14e-01, 1.46e-02, 1.78e-02, 3.05e-01]    [4.79e-02, 5.14e-01, 1.46e-02, 1.78e-02, 3.05e-01]    []  


1000      [5.81e-04, 3.90e-04, 5.65e-05, 4.63e-04, 1.66e-04]    [5.81e-04, 3.90e-04, 5.65e-05, 4.63e-04, 1.66e-04]    []  
2000      [2.91e-04, 9.18e-05, 1.48e-05, 1.35e-04, 9.81e-05]    [2.91e-04, 9.18e-05, 1.48e-05, 1.35e-04, 9.81e-05]    []  
3000      [1.20e-04, 2.48e-05, 5.74e-06, 4.27e-05, 8.81e-05]    [1.20e-04, 2.48e-05, 5.74e-06, 4.27e-05, 8.81e-05]    []  
4000      [5.28e-05, 7.31e-06, 3.13e-06, 5.59e-06, 8.70e-05]    [5.28e-05, 7.31e-06, 3.13e-06, 5.59e-06, 8.70e-05]    []  
5000      [2.23e-05, 3.43e-06, 1.89e-06, 1.72e-06, 8.57e-05]    [2.23e-05, 3.43e-06, 1.89e-06, 1.72e-06, 8.57e-05]    []  

Best model at step 5000:
  train loss: 1.15e-04
  test loss: 1.15e-04
  test metric: []

'train' took 195.053683 s

Learned alpha: 0.10002656280994415
